In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 特徴量リストの定義とモード選択
# ==========================================
function_features = [
    "機能語率", "ひらがな率", "句点頻度", "読点頻度", "助詞割合", "助動詞割合", "連体詞割合", "接続詞割合", "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と", "助詞_か", "助詞_が", "助詞_で", "助詞_や", "助詞_を", "助詞_の",
    "助詞_ほど", "助詞_まで", "助詞_も", "助詞_から", "助詞_たり", "助詞_など", "助詞_ながら", "助詞_だけ", "助詞_ば",
    "助詞_へ", "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら", "助詞_より", "助詞_こそ", "助詞_つつ", "助詞_くらい",
    "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合", "形容詞割合", "副詞割合", "形状詞割合", "漢字率", "カタカナ率", "英数字率"
]

# ★ 実験したい特徴量セットをここで選定 ("All" or "Function" or "Content")
feature_mode = "Content"

if feature_mode == "All":
    base_features = function_features + content_features
elif feature_mode == "Function":
    base_features = function_features
elif feature_mode == "Content":
    base_features = content_features

print(f"【実験設定】選択された特徴量モード: {feature_mode} (ベース特徴量数: {len(base_features)})")

# ==========================================
# 2. データの読み込みと前処理
# ==========================================
current_dir = os.getcwd()
if not current_dir.endswith("AI確信度"):
    current_dir = os.path.join(current_dir, "AI確信度")

doc_csv_path = os.path.join(current_dir, "train_Confidence-Level_one_sentence_stylometry_features.csv")
sent_csv_path = os.path.join(current_dir, "test_Confidence-Level_one_sentence_stylometry_features.csv")

df_doc = pd.read_csv(doc_csv_path).fillna(0)
df_sent = pd.read_csv(sent_csv_path).fillna(0)

# ─── ★修正箇所: 不足している特徴量列を0で自動追加 ───
for col in base_features:
    if col not in df_doc.columns:
        df_doc[col] = 0.0
    if col not in df_sent.columns:
        df_sent[col] = 0.0
# ───────────────────────────────────────────────────

# doc_id と sent_order の整理
df_doc["doc_id"] = df_doc["filename"].str.replace(r"_s\d+$", "", regex=True)
df_doc["sent_order"] = df_doc.groupby("doc_id").cumcount() + 1

df_sent["doc_id"] = df_sent["filename"].str.replace(r"_s\d+$", "", regex=True)
df_sent["sent_order"] = df_sent.groupby("doc_id").cumcount() + 1

# ==========================================
# 3. 文書単位での分割の廃止 (全件をそのまま割り当て)
# ==========================================
train_docs = df_doc["doc_id"].unique()
test_docs = df_sent["doc_id"].unique()

# ==========================================
# 4. 訓練データ（文章単位）の抽出と学習
# ==========================================
df_train = df_doc.copy()

X_train = df_train[base_features].values
y_train = df_train["label"].values

input_dim = X_train.shape[1]
print(f"【学習設定】訓練データ（文章数）: {X_train.shape[0]} 件, 特徴量次元数: {input_dim}")

# 標準化と学習
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ロジスティック回帰モデルの学習
model = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
model.fit(X_train_scaled, y_train)

# ==========================================
# 5. テストデータの予測と確率の累積平均の計算
# ==========================================
results = []

for doc_id in test_docs:
    df_doc_sents = df_sent[df_sent["doc_id"] == doc_id].sort_values("sent_order")
    if df_doc_sents.empty:
        continue
        
    true_label = df_doc_sents["label"].iloc[0]
    proba_history = []
    
    for idx, row in df_doc_sents.iterrows():
        sent_order = row["sent_order"]
        
        # 1. 「その1文だけ」の特徴量を取得して標準化
        feat = row[base_features].values.reshape(1, -1)
        feat_scaled = scaler.transform(feat)
        
        # 2. その1文に対するAI確率を予測
        ai_index = list(model.classes_).index("AI")
        pred_proba_single = model.predict_proba(feat_scaled)[0][ai_index]
        
        # 3. 算出した確率を履歴に追加し、「ここまでの平均確率」を計算
        proba_history.append(pred_proba_single)
        cum_proba_avg = sum(proba_history) / len(proba_history)
        
        # 4. 累積平均確率に基づいて、現時点での暫定ラベルを決定
        pred_label = "AI" if cum_proba_avg >= 0.5 else "Human"
        
        results.append({
            "doc_id": doc_id,
            "sent_order": sent_order,
            "label": true_label,
            "AI確率": cum_proba_avg,
            "予測ラベル": pred_label
        })
# ==========================================
# 特徴量の寄与度（重み・係数）の抽出とCSV保存
# ==========================================
# ロジスティック回帰の係数を取得（1次元配列に変換）
coefficients = model.coef_[0]

# 特徴量名と係数をペアにしたデータフレームを作成
df_importance = pd.DataFrame(
    {
        "特徴量": base_features,
        "回帰係数": coefficients,
        "絶対値": np.abs(coefficients),
        "寄与方向": [
            "AIに寄与 (+)" if c > 0 else "Humanに寄与 (-)"
            for c in coefficients
        ],
    }
)

# 影響度（絶対値）が大きい順にソート
df_importance = df_importance.sort_values(
    by="絶対値", ascending=False
).reset_index(drop=True)

# CSVに保存
importance_filename = f"LR_feature_importance_{feature_mode}.csv"
importance_path = os.path.join(current_dir, importance_filename)
df_importance.to_csv(importance_path, index=False, encoding="utf-8-sig")

print(f"\n[完了] 特徴量の寄与度が '{importance_filename}' に保存されました！\n")
print(df_importance)
# ==========================================
# 6. CSV保存
# ==========================================
df_output = pd.DataFrame(results)
output_filename = f"predict_proba_split_{feature_mode}_results.csv"
output_path = os.path.join(current_dir, output_filename)
df_output.to_csv(output_path, index=False)

print(f"\n[完了] 不足特徴量の0埋め＆分割なしの予測結果が '{output_filename}' に保存されました！")
print(df_output.head(15))

【実験設定】選択された特徴量モード: Content (ベース特徴量数: 9)
【学習設定】訓練データ（文章数）: 721 件, 特徴量次元数: 9

[完了] 特徴量の寄与度が 'LR_feature_importance_Content.csv' に保存されました！

     特徴量      回帰係数       絶対値          寄与方向
0  形状詞割合 -0.495887  0.495887  Humanに寄与 (-)
1  カタカナ率  0.414725  0.414725     AIに寄与 (+)
2   副詞割合  0.349948  0.349948     AIに寄与 (+)
3   名詞割合 -0.238826  0.238826  Humanに寄与 (-)
4   動詞割合 -0.228386  0.228386  Humanに寄与 (-)
5   内容語率 -0.190167  0.190167  Humanに寄与 (-)
6   英数字率  0.150525  0.150525     AIに寄与 (+)
7  形容詞割合 -0.087478  0.087478  Humanに寄与 (-)
8    漢字率  0.049691  0.049691     AIに寄与 (+)

[完了] 不足特徴量の0埋め＆分割なしの予測結果が 'predict_proba_split_Content_results.csv' に保存されました！
                    doc_id  sent_order  label      AI確率  予測ラベル
0   test_Human_text_01.txt           1  Human  0.312846  Human
1   test_Human_text_01.txt           2  Human  0.410426  Human
2   test_Human_text_01.txt           3  Human  0.377713  Human
3   test_Human_text_01.txt           4  Human  0.331025  Human
4   test_Human_text_01.txt           5 

In [9]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 特徴量リストの定義とモード選択
# ==========================================
function_features = [
    "機能語率", "ひらがな率", "句点頻度", "読点頻度", "助詞割合", "助動詞割合", "連体詞割合", "接続詞割合", "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と", "助詞_か", "助詞_が", "助詞_で", "助詞_や", "助詞_を", "助詞_の",
    "助詞_ほど", "助詞_まで", "助詞_も", "助詞_から", "助詞_たり", "助詞_など", "助詞_ながら", "助詞_だけ", "助詞_ば",
    "助詞_へ", "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら", "助詞_より", "助詞_こそ", "助詞_つつ", "助詞_くらい",
    "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合", "形容詞割合", "副詞割合", "形状詞割合", "漢字率", "カタカナ率", "英数字率"
]

# ★ 実験したい特徴量セットを選定 ("All" or "Function" or "Content")
feature_mode = "Content"

if feature_mode == "All":
    base_features = function_features + content_features
elif feature_mode == "Function":
    base_features = function_features
elif feature_mode == "Content":
    base_features = content_features

print(f"【実験設定】選択された特徴量モード: {feature_mode} (ベース特徴量数: {len(base_features)})")

# ==========================================
# 2. データの読み込みと前処理
# ==========================================
current_dir = os.getcwd()
if not current_dir.endswith("AI確信度"):
    current_dir = os.path.join(current_dir, "AI確信度")

doc_csv_path = os.path.join(current_dir, "train_Confidence-Level_one_sentence_stylometry_features.csv")
sent_csv_path = os.path.join(current_dir, "test_Confidence-Level_one_sentence_stylometry_features.csv")

df_doc = pd.read_csv(doc_csv_path).fillna(0)
df_sent = pd.read_csv(sent_csv_path).fillna(0)

# 不足している特徴量列を0で自動追加
for col in base_features:
    if col not in df_doc.columns:
        df_doc[col] = 0.0
    if col not in df_sent.columns:
        df_sent[col] = 0.0

# doc_id と sent_order の整理
df_doc["doc_id"] = df_doc["filename"].str.replace(r"_s\d+$", "", regex=True)
df_doc = df_doc.sort_values(["doc_id", "filename"]) # 順序を揃える
df_doc["sent_order"] = df_doc.groupby("doc_id").cumcount() + 1

df_sent["doc_id"] = df_sent["filename"].str.replace(r"_s\d+$", "", regex=True)
df_sent = df_sent.sort_values(["doc_id", "filename"]) # 順序を揃える
df_sent["sent_order"] = df_sent.groupby("doc_id").cumcount() + 1


# ==========================================
# 3. ★ 特徴量の累積平均化（学習・テスト共通）
# ==========================================
# 各ドキュメントごとに、1文目からその文(sent_order)までの特徴量の累積平均（expanding mean）を算出します。
# これにより、「文章が1文ずつ書き進められている途中の特徴量」を擬似的に表現します。
print("【前処理】特徴量の累積平均（段階的増加）を計算中...")
df_doc[base_features] = df_doc.groupby("doc_id")[base_features].transform(lambda x: x.expanding().mean())
df_sent[base_features] = df_sent.groupby("doc_id")[base_features].transform(lambda x: x.expanding().mean())


# ==========================================
# 4. 訓練データ（段階的プロセスを含む全状態）の抽出と学習
# ==========================================
# 全ての段階（1文だけ、1〜2文目平均、1〜3文目平均...）のデータを学習サンプルとして扱います。
df_train = df_doc.copy()

X_train = df_train[base_features].values
y_train = df_train["label"].values

input_dim = X_train.shape[1]
print(f"【学習設定】訓練データ（総状態数）: {X_train.shape[0]} 件, 特徴量次元数: {input_dim}")

# 標準化と学習
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ロジスティック回帰モデルの学習
model = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
model.fit(X_train_scaled, y_train)


# ==========================================
# 5. テストデータの予測と段階的確率の算出
# ==========================================
test_docs = df_sent["doc_id"].unique()
results = []

for doc_id in test_docs:
    df_doc_sents = df_sent[df_sent["doc_id"] == doc_id].sort_values("sent_order")
    if df_doc_sents.empty:
        continue
    true_label = df_doc_sents["label"].iloc[0]
    
    for idx, row in df_doc_sents.iterrows():
        sent_order = row["sent_order"]
        
        # 1. すでに累積平均化されている特徴量を取得して標準化
        feat = row[base_features].values.reshape(1, -1)
        feat_scaled = scaler.transform(feat)
        
        # 2. 累積特徴量に対するAI確率を予測
        ai_index = list(model.classes_).index("AI")
        pred_proba_cum = model.predict_proba(feat_scaled)[0][ai_index]
        
        # 3. 予測確率に基づいて、現時点での暫定ラベルを決定
        pred_label = "AI" if pred_proba_cum >= 0.5 else "Human"
        
        results.append({
            "doc_id": doc_id,
            "sent_order": sent_order,
            "label": true_label,
            "AI確率": pred_proba_cum,
            "予測ラベル": pred_label
        })

# ==========================================
# 6. CSV保存
# ==========================================
df_output = pd.DataFrame(results)
output_filename = f"more_predict_proba_split_{feature_mode}_results.csv"
output_path = os.path.join(current_dir, output_filename)
df_output.to_csv(output_path, index=False)

print(f"\n[完了] 学習・テスト共に段階的累積特徴量を適用した予測結果が '{output_filename}' に保存されました！")
print(df_output.head(15))

【実験設定】選択された特徴量モード: Content (ベース特徴量数: 9)
【前処理】特徴量の累積平均（段階的増加）を計算中...
【学習設定】訓練データ（総状態数）: 721 件, 特徴量次元数: 9

[完了] 学習・テスト共に段階的累積特徴量を適用した予測結果が 'more_predict_proba_split_Content_results.csv' に保存されました！
                      doc_id  sent_order label      AI確率  予測ラベル
0   test_ChatGPT_text_01.txt           1    AI  0.001521  Human
1   test_ChatGPT_text_01.txt           2    AI  0.003337  Human
2   test_ChatGPT_text_01.txt           3    AI  0.076939  Human
3   test_ChatGPT_text_01.txt           4    AI  0.060501  Human
4   test_ChatGPT_text_01.txt           5    AI  0.021344  Human
5   test_ChatGPT_text_01.txt           6    AI  0.059558  Human
6   test_ChatGPT_text_01.txt           7    AI  0.093257  Human
7   test_ChatGPT_text_01.txt           8    AI  0.114685  Human
8   test_ChatGPT_text_01.txt           9    AI  0.155185  Human
9   test_ChatGPT_text_01.txt          10    AI  0.222657  Human
10  test_ChatGPT_text_02.txt           1    AI  0.671800     AI
11  test_ChatGPT_text_02.txt          

In [2]:
%pip install shap


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import numpy as np
import pandas as pd
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 特徴量の定義
# ==========================================
function_features = [
    "機能語率", "ひらがな率", "句点頻度", "読点頻度",
    "助詞割合", "助動詞割合", "連体詞割合", "接続詞割合",
    "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と", "助詞_か",
    "助詞_が", "助詞_で", "助詞_や", "助詞_を", "助詞_の",
    "助詞_ほど", "助詞_まで", "助詞_も", "助詞_から",
    "助詞_たり", "助詞_など", "助詞_ながら", "助詞_だけ",
    "助詞_ば", "助詞_へ", "助詞_のみ", "助詞_しも",
    "助詞_しか", "助詞_すら", "助詞_より", "助詞_こそ",
    "助詞_つつ", "助詞_くらい", "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合", "形容詞割合",
    "副詞割合", "形状詞割合", "漢字率", "カタカナ率",
    "英数字率"
]

# "All" / "Function" / "Content"
feature_mode = "Content"

if feature_mode == "All":
    base_features = function_features + content_features
elif feature_mode == "Function":
    base_features = function_features
elif feature_mode == "Content":
    base_features = content_features
else:
    raise ValueError("feature_mode の設定が不正です")

# ==========================================
# 2. CSV読み込み
# ==========================================
current_dir = os.getcwd()

if not current_dir.endswith("AI確信度"):
    current_dir = os.path.join(current_dir, "AI確信度")

doc_csv_path = os.path.join(
    current_dir,
    "train_Confidence-Level_one_sentence_stylometry_features.csv"
)

sent_csv_path = os.path.join(
    current_dir,
    "test_Confidence-Level_one_sentence_stylometry_features.csv"
)

df_train = pd.read_csv(doc_csv_path).fillna(0)
df_test = pd.read_csv(sent_csv_path).fillna(0)

# 不足している特徴量列を追加
for col in base_features:
    if col not in df_train.columns:
        df_train[col] = 0.0
    if col not in df_test.columns:
        df_test[col] = 0.0

# 特徴量の欠損値・型を統一
df_train[base_features] = (
    df_train[base_features].astype(float)
)
df_test[base_features] = (
    df_test[base_features].astype(float)
)

# 行順と文書内の文順を確定
df_test = df_test.reset_index(drop=True)

df_test["doc_id"] = (
    df_test["filename"]
    .str.replace(r"_s\d+$", "", regex=True)
)

df_test["sent_order"] = (
    df_test.groupby("doc_id").cumcount() + 1
)

# ==========================================
# 3. LRを学習
# ==========================================
X_train = df_train[base_features].to_numpy()
y_train = df_train["label"].to_numpy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_scaled, y_train)

# AIクラスのインデックス
ai_index = list(model.classes_).index("AI")

# ==========================================
# 4. SHAP Explainerの設定
# ==========================================
def predict_ai_probability(X):
    """元の特徴量からLRのAI確率を返す"""
    X = np.asarray(X, dtype=float)
    X_scaled = scaler.transform(X)
    return model.predict_proba(X_scaled)[:, ai_index]


# 学習データから背景データを抽出
background = (
    df_train[base_features]
    .sample(
        n=min(100, len(df_train)),
        random_state=42
    )
    .to_numpy()
)

masker = shap.maskers.Independent(
    background,
    max_samples=len(background)
)

explainer = shap.Explainer(
    predict_ai_probability,
    masker,
    algorithm="permutation"
)

# ==========================================
# 5. SHAP値のみ計算
# ==========================================
X_test = df_test[base_features].to_numpy()

# Permutation SHAPの評価回数
max_evals = 2 * len(base_features) + 1

shap_result = explainer(
    X_test,
    max_evals=max_evals
)

shap_values = shap_result.values

# ==========================================
# 6. SHAP値をCSV保存
# ==========================================
df_shap = pd.DataFrame(
    shap_values,
    columns=base_features
)

# 文書ID・文順・正解ラベルを付与
df_shap.insert(0, "label", df_test["label"].to_numpy())
df_shap.insert(0, "sent_order", df_test["sent_order"].to_numpy())
df_shap.insert(0, "doc_id", df_test["doc_id"].to_numpy())

output_filename = f"SHAP_values_{feature_mode}.csv"
output_path = os.path.join(current_dir, output_filename)

df_shap.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"SHAP計算完了: {output_path}")
print(f"対象文数: {len(df_test)}")
print(f"特徴量数: {len(base_features)}")
print(f"SHAP配列の形状: {shap_values.shape}")

SHAP計算完了: /workspace/研究/one-sentence/AI確信度/SHAP_values_Content.csv
対象文数: 206
特徴量数: 9
SHAP配列の形状: (206, 9)
